# Étape 6 : règles de nettoyage

Ce notebook applique aux données chargées des règles de nettoyage explicites, dans un ordre fixe, sans jamais modifier les fichiers bruts : il travaille sur une copie, et chaque ligne garde son lignage. Chaque modification est inscrite dans un journal des corrections (table, ligne source, colonne, règle, valeur avant et après). Une valeur que la règle ne sait pas corriger sans ambiguïté reste telle quelle et reste une anomalie. Les valeurs des colonnes sensibles ne sont jamais affichées.

Chaque mois : mettre à jour `PERIODE_FIN`, exécuter d'abord le notebook `03_dictionnaire`, puis celui-ci en entier. Après l'arbitrage, enregistrer et fermer le classeur, puis relancer le notebook en entier : la cellule `# IMPORT ARBITRAGE`, placée avant `# NETTOYAGE`, enregistre les décisions, qui s'appliquent dans la même exécution.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from frigeo import racine_projet
from frigeo.perimetre import decouvrir

# PARAMETRE A MODIFIER CHAQUE MOIS
# Dernier mois déposé dans data/raw, au format AAAA-MM
PERIODE_FIN = "2026-03"

In [ ]:
%%time
# CHARGEMENT

from frigeo.chargement import charger_tout
from frigeo.controles import charger_dictionnaire_enregistre
from frigeo.dictionnaire import charger_meta

DATA = racine_projet() / "data"
perimetre = decouvrir(DATA, PERIODE_FIN)
perimetre.verifier()
print(perimetre.resume())

tables = charger_tout(perimetre)
# Sans dictionnaire enregistré, le message renvoie au notebook 03_dictionnaire.
dictionnaire = charger_dictionnaire_enregistre()
print(len(tables), "tables,", sum(len(c) for c in dictionnaire.values()), "colonnes au dictionnaire")
print("dictionnaire de la période", charger_meta()["periode_fin"], ", données jusqu'à", PERIODE_FIN)

In [ ]:
# IMPORT ARBITRAGE

from frigeo.decisions import (
    charger_decisions,
    chemin_arbitrage,
    ecrire_decisions,
    importer_arbitrage,
    repere_arbitrage,
)
from frigeo.formulaires import ecrire_et_archiver, exiger_variables

exiger_variables(globals(), ["PERIODE_FIN", "dictionnaire"],
                 "la cellule des paramètres, puis # CHARGEMENT")

# Placée avant # NETTOYAGE : à chaque exécution complète du notebook, le classeur que le
# métier a rempli, enregistré puis fermé dans Excel est importé, et ses décisions
# s'appliquent dans la même exécution. Sans classeur, ou avec un classeur déjà importé,
# elle ne fait rien. Tous les problèmes sont listés en une fois, et rien n'est
# enregistré s'il y en a.
chemin_classeur = chemin_arbitrage(PERIODE_FIN)
nb_imports = len(charger_decisions()["meta"]["imports"])
if not chemin_classeur.exists():
    print("Aucun classeur à importer dans", chemin_classeur.parent)
elif repere_arbitrage(chemin_classeur) != nb_imports:
    print("Le classeur", chemin_classeur.name, "est déjà importé ou périmé, rien n'est lu.")
else:
    arbitrage = importer_arbitrage(chemin_classeur, dictionnaire, nb_imports=nb_imports)
    print(len(arbitrage.decisions), "décisions lues dans", chemin_classeur.name)
    if len(arbitrage.decisions):
        display(pd.crosstab(arbitrage.decisions["sorte"], arbitrage.decisions["decision"])
                .rename_axis(index="sorte", columns="decision"))
    print(len(arbitrage.avertissements), "avertissements")
    for message in arbitrage.avertissements:
        print("  ", message)
    if arbitrage.decisions.empty:
        print("Aucune décision, rien n'est enregistré (le classeur reste importable).")
    else:
        # Le classeur est d'abord archivé dans data/formulaires/appliques (le fermer dans
        # Excel), puis les décisions sont enregistrées.
        (fichier, entree), archive = ecrire_et_archiver(
            chemin_classeur, lambda: ecrire_decisions(arbitrage, PERIODE_FIN)
        )
        print(entree["decisions"], f"décisions enregistrées dans {fichier}, dont",
              entree["remplacees"], "remplacent une décision caduque")
        print("classeur archivé dans", archive)

In [ ]:
# REGLES DE NETTOYAGE

from frigeo.nettoyage import catalogue_nettoyage

# Règles appliquées, dans leur ordre, avec ce qu'elles corrigent et ce qu'elles ne
# corrigent jamais.
display(catalogue_nettoyage())

In [ ]:
%%time
# NETTOYAGE

from frigeo.decisions import charger_decisions, chemin_decisions
from frigeo.nettoyage import (
    ACTION_MODIFIEE,
    ACTION_RETIREE,
    charger_cle_empreintes,
    nettoyer,
)

# Les tables chargées ne sont pas modifiées, tables_nettoyees en est une copie
# corrigée. Le journal porte des valeurs du client, il n'est pas affiché.
# La date de coupure (fin de la période extraite) borne les années écrites sur deux chiffres.
# Les décisions du métier (data/config/decisions.yaml) s'appliquent en dernier.
decisions_metier = charger_decisions()
print(len(decisions_metier["decisions"]), "décisions du métier lues dans", chemin_decisions())
# Une valeur d'une colonne sensible est masquée au journal et remplacée par son
# empreinte, calculée avec une clé secrète (data/config/cle_empreintes.txt, créée au
# premier nettoyage), à garder avec les données du client, hors du dépôt.
cle_empreintes = charger_cle_empreintes()
print("clé des empreintes", "créée dans" if cle_empreintes.creee else "lue dans",
      cle_empreintes.chemin)
# Un code que le métier a rattaché à un code principal reste tel quel dans les tables
# nettoyées : la correspondance est écrite en base par # BASE (qualite.correspondance_cle).
tables_nettoyees, journal, bilan_nettoyage, decisions_caduques, correspondances_codes = (
    nettoyer(tables, dictionnaire, date_coupure=perimetre.date_coupure,
             decisions=decisions_metier, cle_empreinte=cle_empreintes.cle)
)
print(int((journal["action"] == ACTION_MODIFIEE).sum()), "cellules corrigées,",
      int((journal["action"] == ACTION_RETIREE).sum()), "lignes retirées,",
      int(bilan_nettoyage["non_corrigees"].sum()), "cellules, lignes ou codes visés non corrigés")
print(len(correspondances_codes), "codes en doublon regroupés sous un code principal")
display(bilan_nettoyage)

In [ ]:
# COLONNES SENSIBLES

from frigeo.confidentialite import charger_sensibilite, est_sensible
from frigeo.controles import VALEUR_MASQUEE
from frigeo.dictionnaire import est_masquee
from frigeo.doublons import charger_declarations_doublons
from frigeo.nettoyage import empreintes_incoherentes

sensibles = charger_sensibilite()
declarations = charger_declarations_doublons()


def sensible(table, colonne):
    regles = dictionnaire[table][colonne]
    return (est_sensible(sensibles, table, colonne) or est_masquee(regles["valeurs"])
            or est_masquee(regles["format"]))


# Les valeurs ne figurent que sur les cellules modifiées (une ligne retirée n'en a pas).
modifiees = journal[journal["action"] == ACTION_MODIFIEE]
sur_sensible = [sensible(t, c) for t, c in zip(modifiees["table"], modifiees["colonne"])]
cle_sensible = [
    any(est_sensible(sensibles, t, c) for c in declarations.get(t, {}).get("cle", []))
    for t in journal["table"]
]
masquees = ((modifiees["valeur_avant"] == VALEUR_MASQUEE)
            & (modifiees["valeur_apres"] == VALEUR_MASQUEE))
print("corrections sur une colonne sensible", sum(sur_sensible),
      ", dont valeurs non masquées (0 attendu)", int((~masquees[sur_sensible]).sum()))
# Une valeur masquée porte ses deux empreintes, toute autre ligne n'en porte aucune.
avec_empreintes = modifiees["empreinte_avant"].notna() & modifiees["empreinte_apres"].notna()
print("corrections sur une colonne sensible avec leurs deux empreintes",
      int(avec_empreintes[sur_sensible].sum()), "sur", sum(sur_sensible))
print("lignes du journal aux empreintes incohérentes (0 attendu)",
      len(empreintes_incoherentes(journal)))
print("lignes du journal dont la clé a une colonne sensible", sum(cle_sensible),
      ", dont clés non masquées (0 attendu)",
      int((journal.loc[cle_sensible, "cle"] != VALEUR_MASQUEE).sum()))

In [ ]:
# DECISIONS CADUQUES

from frigeo.decisions import retirer_decisions
from frigeo.formulaires import exiger_variables

exiger_variables(globals(), ["PERIODE_FIN", "decisions_metier", "decisions_caduques"],
                 "la cellule des paramètres, puis # CHARGEMENT et # NETTOYAGE")

# Une décision caduque (« corriger », « retirer » ou « rattacher ») n'a pas été appliquée
# parce que sa donnée a changé depuis la décision : elle ne s'appliquera plus. Pour
# retirer les décisions caduques de data/config/decisions.yaml (copie datée dans
# data/config/sauvegardes), mettre RETIRER_CADUQUES à True et relancer cette cellule,
# puis remettre False et relancer # NETTOYAGE.
RETIRER_CADUQUES = False

print(len(decisions_caduques), "décisions caduques")
if len(decisions_caduques):
    display(decisions_caduques.groupby(["table", "champ", "decision"], dropna=False)
            .size().rename("decisions").reset_index())
if len(decisions_caduques) and RETIRER_CADUQUES:
    fichier, entree = retirer_decisions(
        decisions_caduques["id_anomalie"], PERIODE_FIN, decisions_metier
    )
    print(entree["decisions"], "décisions retirées de", fichier)
    print("Remettre RETIRER_CADUQUES à False, puis relancer # NETTOYAGE.")
elif len(decisions_caduques):
    print("Pour les retirer, mettre RETIRER_CADUQUES à True puis relancer cette cellule.")

In [ ]:
%%time
# ANOMALIES AVANT NETTOYAGE

from frigeo.anomalies import (
    bilan_controles,
    charger_qualification,
    lignes_non_evaluables,
    qualifier,
    reunir_anomalies,
)
from frigeo.controles import charger_regles_complementaires, controler
from frigeo.doublons import charger_declarations_doublons, detecter_doublons
from frigeo.gestion import charger_regles_gestion, controler_gestion

# Les contrôles des étapes 3 et 4 sur les tables chargées, avant nettoyage, comme au
# notebook 06, pour mesurer ce que le nettoyage résout (cellule # COMPARAISON).
declarations = charger_declarations_doublons()
ecarts_avant, bilan_avant = controler(
    tables, dictionnaire, complementaires=charger_regles_complementaires()
)
doublons_avant, bilan_doublons_avant = detecter_doublons(tables, declarations)
ecarts_gestion_avant, bilan_gestion_avant, non_evaluables_gestion_avant = controler_gestion(
    tables, dictionnaire, charger_regles_gestion(), declarations, doublons_avant,
    date_coupure=perimetre.date_coupure, avec_non_evaluables=True,
)
anomalies_avant = reunir_anomalies(
    tables, ecarts_avant, doublons_avant, ecarts_gestion_avant,
    bilan_avant, bilan_doublons_avant, bilan_gestion_avant, declarations,
)
# Gravité et indicateurs, comme au notebook 06 : la table et son bilan sont ceux du
# fichier anomalies_AAAA-MM.csv, écrits en base à l'état « brut » (cellule # BASE).
bilan_anomalies_avant = bilan_controles(
    bilan_avant, bilan_doublons_avant, bilan_gestion_avant, anomalies_avant, declarations
)
anomalies_avant, bilan_anomalies_avant, _ = qualifier(
    anomalies_avant, bilan_anomalies_avant, charger_qualification(), tables
)
non_evaluables_avant = lignes_non_evaluables(tables, non_evaluables_gestion_avant)
print(len(anomalies_avant), "lignes dans la table des anomalies avant nettoyage,",
      "comme au notebook 06")
print(len(non_evaluables_avant), "lignes non évaluables des règles de gestion avant nettoyage")

In [ ]:
%%time
# ANOMALIES APRES NETTOYAGE

from frigeo.anomalies import (
    anomalies_non_masquees,
    bilan_controles,
    charger_qualification,
    lignes_non_evaluables,
    qualifier,
    reunir_anomalies,
)
from frigeo.controles import charger_regles_complementaires, controler
from frigeo.doublons import charger_declarations_doublons, detecter_doublons
from frigeo.gestion import charger_regles_gestion, controler_gestion

# Les contrôles des étapes 3 et 4 et la table des anomalies de l'étape 5, relancés
# sur les tables nettoyées. Ce qui reste est ce que les règles automatiques n'ont
# pas résolu, et sert au classeur d'arbitrage. Décomptes seulement.
declarations = charger_declarations_doublons()
ecarts_apres, bilan_apres = controler(
    tables_nettoyees, dictionnaire, complementaires=charger_regles_complementaires()
)
doublons_apres, bilan_doublons_apres = detecter_doublons(tables_nettoyees, declarations)
ecarts_gestion_apres, bilan_gestion_apres, non_evaluables_gestion_apres = controler_gestion(
    tables_nettoyees, dictionnaire, charger_regles_gestion(), declarations, doublons_apres,
    date_coupure=perimetre.date_coupure, avec_non_evaluables=True,
)
anomalies_apres = reunir_anomalies(
    tables_nettoyees, ecarts_apres, doublons_apres, ecarts_gestion_apres,
    bilan_apres, bilan_doublons_apres, bilan_gestion_apres, declarations,
)
bilan_anomalies_apres = bilan_controles(
    bilan_apres, bilan_doublons_apres, bilan_gestion_apres, anomalies_apres, declarations
)
anomalies_apres, bilan_anomalies_apres, _ = qualifier(
    anomalies_apres, bilan_anomalies_apres, charger_qualification(), tables_nettoyees
)
non_evaluables_apres = lignes_non_evaluables(tables_nettoyees, non_evaluables_gestion_apres)
print(len(anomalies_apres), "lignes dans la table des anomalies après nettoyage,",
      anomalies_apres["id_anomalie"].nunique(), "identifiants distincts")
display(anomalies_apres.groupby(["etape", "issue"], sort=False).size().rename("lignes").to_frame())
print(len(anomalies_non_masquees(anomalies_apres, declarations=declarations)),
      "lignes dont la valeur ou la clé sensible n'est pas masquée (0 attendu)")
print(len(non_evaluables_apres), "lignes non évaluables des règles de gestion après nettoyage")

In [ ]:
# COMPARAISON

from frigeo.nettoyage import comparer_anomalies

# Par contrôle, ce que le nettoyage et les décisions du métier ont résolu (identifiant
# disparu), fait apparaître (identifiant nouveau) ou changé d'issue. Décomptes seulement.
comparaison = comparer_anomalies(anomalies_avant, anomalies_apres)
print(len(comparaison), "contrôles avec au moins un écart avant ou après nettoyage")
display(comparaison.groupby("etape", sort=False)[list(comparaison.columns[4:])].sum())
# Une apparition n'est pas forcément un défaut, une valeur devenue lisible rend
# évaluable une règle qui ne l'était pas. Ces contrôles sont à regarder un par un.
a_regarder = comparaison[(comparaison["apparues"] > 0) | (comparaison["issue_changee"] > 0)]
print(len(a_regarder), "contrôles avec des écarts apparus ou une issue changée")
if len(a_regarder):
    display(a_regarder)
# Lignes non évaluables par règle de gestion, avant et après nettoyage : une valeur
# corrigée rend évaluable la ligne qui ne l'était pas.
non_evaluables_avant_apres = pd.concat(
    [
        non_evaluables_avant.groupby("controle", sort=False).size().rename("avant"),
        non_evaluables_apres.groupby("controle", sort=False).size().rename("apres"),
    ],
    axis=1,
).fillna(0).astype(int)
print(len(non_evaluables_avant_apres),
      "règles de gestion avec des lignes non évaluables avant ou après nettoyage")
display(non_evaluables_avant_apres)

In [ ]:
# EXPORT ARBITRAGE

from frigeo.decisions import (
    ETATS_A_ARBITRER,
    charger_decisions,
    chemin_arbitrage,
    chemin_decisions,
    elements_a_arbitrer,
    exporter_arbitrage,
    repere_arbitrage,
)
from frigeo.formulaires import exiger_variables

exiger_variables(globals(), ["PERIODE_FIN", "tables_nettoyees", "anomalies_apres"],
                 "les cellules précédentes, jusqu'à # ANOMALIES APRES NETTOYAGE")

# Éléments que les règles automatiques ne savent pas trancher, soumis au métier dans
# un classeur Excel (data/formulaires). Le tableau des éléments porte des valeurs du
# client, il n'est pas affiché.
decisions = charger_decisions()
nb_imports = len(decisions["meta"]["imports"])
print(len(decisions["decisions"]), "décisions enregistrées,", nb_imports, "imports, lus dans",
      chemin_decisions())
elements = elements_a_arbitrer(tables_nettoyees, anomalies_apres, decisions)
display(pd.crosstab(elements["sorte"], elements["etat_decision"])
        .rename_axis(index="sorte", columns="etat_decision"))
a_exporter = int(elements["etat_decision"].isin(ETATS_A_ARBITRER).sum())
chemin_classeur = chemin_arbitrage(PERIODE_FIN)
if a_exporter == 0:
    print("Aucun élément à arbitrer, pas de classeur.")
elif chemin_classeur.exists() and repere_arbitrage(chemin_classeur) == nb_imports:
    print("Le classeur", chemin_classeur.name, "attend les décisions du métier, puis une",
          "nouvelle exécution complète du notebook (# IMPORT ARBITRAGE l'importe).")
elif chemin_classeur.exists():
    print("Le classeur", chemin_classeur.name, "est déjà importé ou périmé. Le supprimer",
          "(Explorateur Windows, dossier data\\formulaires), puis relancer cette cellule.")
else:
    exporter_arbitrage(elements, PERIODE_FIN, nb_imports=nb_imports)
    print(a_exporter, "éléments écrits dans", chemin_classeur)

In [ ]:
# ECRITURE

from frigeo.anomalies import chemin_non_evaluables, dossier_anomalies, ecrire_anomalies
from frigeo.decisions import ajouter_decisions
from frigeo.nettoyage import dossier_nettoyage, ecrire_nettoyage

# Réécrit à chaque exécution, sans copie, le journal et le bilan du nettoyage
# (data/nettoyage), puis la table des anomalies après nettoyage, avec la décision du
# métier, son bilan et les lignes non évaluables (data/anomalies, à côté des fichiers
# du notebook 06).
# En cas de refus (valeur sensible visible, table non qualifiée), rien n'est écrit.
# Si un des fichiers est ouvert dans Excel, le fermer puis relancer cette cellule.
chemin_journal, chemin_bilan_nettoyage = ecrire_nettoyage(
    journal, bilan_nettoyage, PERIODE_FIN, declarations=declarations
)
print("fichiers écrits dans", dossier_nettoyage())
print("  ", f"{chemin_journal.name},", len(journal), "lignes")
print("  ", f"{chemin_bilan_nettoyage.name},", len(bilan_nettoyage), "lignes")
anomalies_nettoyees = ajouter_decisions(anomalies_apres, elements, decisions)
chemin_anomalies, chemin_bilan = ecrire_anomalies(
    anomalies_nettoyees, bilan_anomalies_apres, PERIODE_FIN, declarations=declarations,
    nettoyees=True, non_evaluables=non_evaluables_apres,
)
print("fichiers écrits dans", dossier_anomalies())
print("  ", f"{chemin_anomalies.name},", len(anomalies_nettoyees), "lignes, dont",
      int(anomalies_nettoyees["decision_metier"].notna().sum()), "avec une décision du métier")
print("  ", f"{chemin_bilan.name},", len(bilan_anomalies_apres), "contrôles")
print("  ", f"{chemin_non_evaluables(PERIODE_FIN, nettoyees=True).name},",
      len(non_evaluables_apres), "lignes")

In [ ]:
# BASE

from frigeo.anomalies import lignes_non_retenues
from frigeo.base import creer_moteur
from frigeo.ecriture_base import CompleterQualite, SortiesQualite, ecrire_base
from frigeo.formulaires import exiger_variables
from frigeo.gestion import charger_jours_feries, charger_regles_gestion
from frigeo.rattachement import rattacher

exiger_variables(globals(), ["PERIODE_FIN", "perimetre", "tables", "tables_nettoyees",
                             "dictionnaire", "anomalies_avant", "anomalies_nettoyees",
                             "correspondances_codes"],
                 "toutes les cellules précédentes, jusqu'à # ECRITURE")

# Écrit l'exécution dans la base du fichier .env, construite par python -m frigeo.base :
# tables chargement (valeurs telles que livrées) et propre (nettoyées et typées),
# supprimées et recréées à chaque exécution, et dans qualite les fichiers chargés, la
# correspondance des colonnes, les anomalies avant (état « brut ») et après nettoyage
# (état « nettoyé »), leurs bilans et lignes non évaluables, le journal et le bilan du
# nettoyage, les décisions du métier appliquées, et les tables de référence issues de
# la configuration (jours fériés de config/gestion.yaml, valeurs admises des listes
# fermées du dictionnaire, hors colonnes sensibles), ainsi que le rattachement des
# lignes nettoyées à leurs référentiels et les codes en doublon regroupés sous un code
# principal par les décisions du métier. Si un pack de config/base.yaml
# construit dw, dw.reconstruire est appelée en dernier, dans la même transaction : un
# contrat d'entrée non respecté fait échouer l'exécution. Les valeurs sont celles des fichiers
# CSV (une valeur sensible reste « masqué »), avec les mêmes refus. Rejouer la même
# PERIODE_FIN remplace l'exécution précédente de cette période. En cas d'erreur ou de
# refus, rien n'est écrit et l'exécution est inscrite « échouée ». Décomptes seulement.
# Rattachement de chaque ligne nettoyée qui cite un référentiel à la ligne qu'elle cite,
# par les règles de config/gestion.yaml qui rapprochent une clé d'un référentiel (avec
# leur comparaison, sans leur filtre) : dw s'en sert pour relier chaque fait au bon
# membre de dimension. Le bilan par rattachement est affiché après l'écriture.
# Lignes retenues pour les indicateurs : une ligne nettoyée qui porte une anomalie
# bloquante (gravité de config/anomalies.yaml) sans décision du métier reste dans propre
# et dans dw, avec retenue à 0, et sera exclue des indicateurs. Décompte par table
# affiché après l'écriture.
rattachements = rattacher(tables_nettoyees, dictionnaire, charger_regles_gestion())
non_retenues = lignes_non_retenues(anomalies_nettoyees)
sorties = SortiesQualite(
    anomalies_brutes=anomalies_avant,
    bilan_brut=bilan_anomalies_avant,
    non_evaluables_brutes=non_evaluables_avant,
    anomalies_nettoyees=anomalies_nettoyees,
    bilan_nettoye=bilan_anomalies_apres,
    non_evaluables_nettoyees=non_evaluables_apres,
    journal_nettoyage=journal,
    bilan_nettoyage=bilan_nettoyage,
    decisions=decisions_metier,
    dictionnaire=dictionnaire,
    cle_empreinte=cle_empreintes.cle,
    jours_feries=charger_jours_feries(),
    rattachements=rattachements,
    correspondances=correspondances_codes,
)
qualite = CompleterQualite(sorties, PERIODE_FIN, declarations=declarations)
moteur = creer_moteur()
try:
    ecriture = ecrire_base(moteur, PERIODE_FIN, perimetre, tables, tables_nettoyees,
                           completer=qualite, non_retenues=non_retenues)
finally:
    moteur.dispose()
print("exécution", ecriture.id_execution, "réussie,", ecriture.fichiers,
      "fichiers ou feuilles chargés")
display(ecriture.lignes.set_index("table"))
display(pd.Series(qualite.comptes, name="lignes").rename_axis("table de qualite").to_frame())
# Par rattachement : lignes de la table, à clé vide (membre « non renseigné » de la
# dimension), à clé introuvable dans le référentiel (membre « introuvable »), rattachées,
# rattachées parmi plusieurs lignes du référentiel, et parmi elles choisies par la date.
display(rattachements.bilan.set_index("regle"))
print(len(non_retenues), "lignes non retenues (anomalie bloquante non résolue)")
if non_retenues:
    display(pd.Series([cle[0] for cle in non_retenues], name="lignes non retenues")
            .value_counts().rename_axis("table").to_frame())
# Une valeur restée illisible après nettoyage est écrite vide dans propre, et
# reste une anomalie de la table des anomalies après nettoyage.
print(int(ecriture.illisibles["cellules"].sum()), "cellules illisibles écrites vides dans propre")
if len(ecriture.illisibles):
    display(ecriture.illisibles)
if ecriture.journal_dw is None:
    print("dw non reconstruit : aucun pack ne déclare la procédure dw.reconstruire")
else:
    print("dw reconstruit pour l'exécution", ecriture.id_execution)
    display(pd.DataFrame(ecriture.journal_dw, columns=["ordre", "étape", "lignes"])
            .set_index("ordre"))

In [ ]:
# VERIFICATION BASE

from frigeo.base import creer_moteur
from frigeo.ecriture_base import verifier_base
from frigeo.formulaires import exiger_variables

exiger_variables(globals(), ["PERIODE_FIN", "ecriture", "sorties"], "la cellule # BASE")

# Relit la base et compare avec ce que Python a préparé : lignes et cellules renseignées
# de chaque table de chargement et de propre, sommes exactes, dates et heures extrêmes,
# cellules illisibles rattachées à une anomalie après nettoyage, lignes non retenues de
# propre et de chaque fait égales, ligne à ligne, à celles qui portent une anomalie
# bloquante non résolue (ou dont l'en-tête n'est pas retenu), lignes de chaque table
# de qualite par état (jours fériés, valeurs des listes fermées et rattachements
# compris), lignes rattachées par rattachement, journal de la reconstruction de dw et
# contrat d'entrée du pack, lignes de chaque fait et membres -1 et -2 (ou liens non
# renseignés et introuvables) comparés au bilan des rattachements, exécution unique de
# la période.
# Décomptes et agrégats seulement, ceux d'une colonne sensible masqués. Rien n'est
# modifié en base.
moteur = creer_moteur()
try:
    verification = verifier_base(moteur, ecriture, PERIODE_FIN, sorties)
finally:
    moteur.dispose()
display(verification.groupby("famille", sort=False)["conforme"]
        .agg(controles="size", conformes="sum"))
non_conformes = verification[~verification["conforme"]]
print(len(non_conformes), "contrôles non conformes (0 attendu)")
if len(non_conformes):
    display(non_conformes)